# A minimalistic Echo State Networks demo

For the entire story see: http://www.scholarpedia.org/article/Echo_state_network

Based on code by  Mantas
http://mantas.info

# Initialization

In [ ]:

from numpy import *
from matplotlib.pyplot import *
import scipy.linalg


# Prepare Data

In [ ]:
# training data
trainLen = 2000
testLen = 2000
initLen = 100

# create some data (that we want the network to produce)
data = sin(linspace(0,399*pi,5000)) + sin(linspace(0,203*pi,5000))

# plot some of it
figure()
plot(data[0:1000])
title('A sample of data')


# Make Reservoir

In [ ]:
# Size of Input and Output Layers
inSize = outSize = 1

# generate the ESN reservoir

# reservoir size (number of neurons)
resSize = 100

# leaking rate (mixing)
a = 0.3

random.seed(42)

Win = (random.rand(resSize,1+inSize)-0.5) * 1
W = random.rand(resSize,resSize)-0.5

# normalizing and setting spectral radius (correct, slow):
print('Computing spectral radius...'),
rhoW = max(abs(linalg.eig(W)[0]))

print('spectral radius is '+str(rhoW))

W *= 1.25 / rhoW

# Run the Network

In [ ]:
Yt = data[None,initLen+1:trainLen+1]

In [ ]:
# we give the network a constant input with a single neuron:
input_0 = 1

# allocated memory for the design (collected states) matrix
X = zeros((1+inSize+resSize,trainLen-initLen))

# run the reservoir with the data (input) and collect output (X)

# initialize reservoir state
x = zeros((resSize,1))

for t in range(trainLen):
    u = data[t]
    # x = tanh( dot( Win, vstack((input_0,u)) ) + dot( W, x ) )

    # with a leaky neuron
    x = (1-a)*x + a*tanh( dot( Win, vstack((input_0,u)) ) + dot( W, x ) )

    if t >= initLen:
        X[:,t-initLen] = vstack((input_0,u,x))[:,0]

# (this will also be the corresponding target matrix )

figure(figsize=(15,4))
pcolor(X)
xlabel('time (steps)')
ylabel('neurons')

# Train the Network

In [ ]:
# train the output by ridge regression (a more robust variant of linear regression)
# for the usual regression simply set reg to 0.
reg = 1e-8  # regularization coefficient
X_T = X.T

# one line learning:
Wout = dot( dot(Yt,X_T), linalg.inv( dot(X,X_T) + \
    reg*eye(outSize+inSize+resSize) ) )


figure(2, figsize=(1,8))
pcolor(Wout.T)
colorbar()


# Regenerate the Data

In [ ]:
# run the trained ESN in a generative mode. no need to initialize here,
# because x was initialized with training data and we continue from there.

Y = zeros((outSize,testLen)) #allocate output

u = data[trainLen]
for t in range(testLen):
    x = (1-a)*x + a*tanh( dot( Win, vstack((input_0,u)) ) + dot( W, x ) )
    y = dot( Wout, vstack((input_0,u,x)) )
    Y[:,t] = y
    # generative mode:
    u = y
    ## this would be a predictive mode:
    # u = data[trainLen+t+1]

# compute MSE for the first errorLen time steps
errorLen = 500
mse = sum( square( data[trainLen+1:trainLen+errorLen+1] -
    Y[0,0:errorLen] ) ) / errorLen
print('Mean Square Error (MSE) = ' + str( mse ))



In [ ]:
# plot signals
figure(1, figsize=(15,4))

plot( Y.T, 'b' )
plot( data[trainLen+1:trainLen+testLen+1], 'g' )
title('Target and generated signals $y(n)$ starting at $n=0$')
legend(['Target signal', 'Free-running predicted signal'])

figure(2, figsize=(15,8))
pcolor( X[0:100,0:2000])
title('Some reservoir activations $\mathbf{x}(n)$')

# figure(3).clear()
# bar( range(1+inSize+resSize), Wout.T )
# title('Output weights $\mathbf{W}^{out}$')
